In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

In [ ]:
CANDIDATES = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]
DATA_DIR = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, 'train')):
        DATA_DIR = c
        break
if DATA_DIR is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train' in dirs:
            DATA_DIR = root
            break
if DATA_DIR is None:
    DATA_DIR = '../data'

TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB = os.path.join(DATA_DIR, 'sample_submission.csv')
print(f'DATA_DIR: {DATA_DIR}')

FORMATIONS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

In [ ]:
# ===== 診断: 訓練データで formation_tvt の精度を確認 =====
print('=== Training diagnostic: formation_tvt accuracy ===')
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
rmse_list = []
nan_count = 0

for f in train_files[:50]:  # 最初の50ウェルで確認
    hw = pd.read_csv(f)
    if not any(col in hw.columns for col in FORMATIONS):
        continue
    anchor    = hw[hw['TVT_input'].notna()]
    eval_zone = hw[hw['TVT_input'].isna() & hw['TVT'].notna()]
    if len(eval_zone) == 0 or len(anchor) == 0:
        continue
    last = anchor.iloc[-1]
    la_tvt  = float(last['TVT_input'])
    la_z    = float(last['Z'])
    la_ancc = float(last['ANCC'])
    if not np.isfinite(la_ancc) or la_ancc == 0.0:
        nan_count += 1
        continue
    b = la_tvt + la_z - la_ancc
    f_tvt = b - eval_zone['Z'].values + eval_zone['ANCC'].values
    valid = np.isfinite(f_tvt)
    if valid.sum() == 0:
        nan_count += 1
        continue
    rmse = np.sqrt(np.mean((eval_zone['TVT'].values[valid] - f_tvt[valid])**2))
    rmse_list.append(rmse)

print(f'Wells checked: {len(rmse_list)}, ANCC invalid: {nan_count}')
print(f'formation_tvt RMSE: mean={np.mean(rmse_list):.4f}, median={np.median(rmse_list):.4f}')
print(f'RMSE < 1ft: {sum(r < 1 for r in rmse_list)}/{len(rmse_list)} wells')

In [ ]:
# ===== 診断: テストウェルの formation 列 lookup を確認 =====
print('=== Test well formation lookup diagnostic ===')
sub = pd.read_csv(SAMPLE_SUB)
test_well_ids = sorted(sub['id'].str.rsplit('_', n=1).str[0].unique())
print(f'Test wells: {test_well_ids}')

for well_id in test_well_ids:
    print(f'\n--- {well_id} ---')
    hw_test = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))
    print(f'Test CSV columns: {list(hw_test.columns)}')
    print(f'Test CSV rows: {len(hw_test)}, MD range: [{hw_test["MD"].min():.1f}, {hw_test["MD"].max():.1f}]')

    has_form = any(c in hw_test.columns for c in FORMATIONS)
    print(f'Has formation cols in test CSV: {has_form}')

    # training から lookup
    train_path = os.path.join(TRAIN_DIR, f'{well_id}__horizontal_well.csv')
    if os.path.exists(train_path):
        hw_tr = pd.read_csv(train_path)
        print(f'Train CSV rows: {len(hw_tr)}, MD range: [{hw_tr["MD"].min():.1f}, {hw_tr["MD"].max():.1f}]')
        print(f'Train CSV has ANCC: {"ANCC" in hw_tr.columns}')

        # MD がどれだけマッチするか確認
        merged = hw_test[['MD']].merge(hw_tr[['MD'] + FORMATIONS], on='MD', how='left')
        n_matched = merged['ANCC'].notna().sum()
        print(f'MD exact merge: {n_matched}/{len(hw_test)} rows matched')

        # nearest-neighbor で確認
        test_md = hw_test['MD'].values
        train_md = hw_tr['MD'].values
        idx = np.searchsorted(train_md, test_md).clip(0, len(train_md)-1)
        ancc_nn = hw_tr['ANCC'].values[idx] if 'ANCC' in hw_tr.columns else None
        if ancc_nn is not None:
            print(f'Nearest-neighbor ANCC sample: {ancc_nn[:5]}')
            print(f'NaN in NN ANCC: {np.isnan(ancc_nn).sum()}')
    else:
        print('Train CSV NOT FOUND')

In [ ]:
# ===== formation_tvt のみで submission 作成 =====
print('\n=== Generating formation-only submission ===')
all_preds = {}

for well_id in test_well_ids:
    hw_test = pd.read_csv(os.path.join(TEST_DIR, f'{well_id}__horizontal_well.csv'))

    # formation 列を training から nearest-neighbor で補完
    train_path = os.path.join(TRAIN_DIR, f'{well_id}__horizontal_well.csv')
    hw_tr = pd.read_csv(train_path)

    # まず exact merge を試みる
    merged = hw_test.merge(hw_tr[['MD'] + FORMATIONS], on='MD', how='left')
    n_exact = merged['ANCC'].notna().sum()

    if n_exact < len(hw_test) * 0.5:
        # exact merge が半分以上失敗 → nearest-neighbor にフォールバック
        print(f'{well_id}: exact merge {n_exact}/{len(hw_test)} → using nearest-neighbor')
        train_md = hw_tr['MD'].values
        test_md  = hw_test['MD'].values
        idx = np.searchsorted(train_md, test_md).clip(0, len(train_md)-1)
        for col in FORMATIONS:
            if col in hw_tr.columns:
                hw_test[col] = hw_tr[col].values[idx]
    else:
        print(f'{well_id}: exact merge {n_exact}/{len(hw_test)} → using exact merge')
        for col in FORMATIONS:
            hw_test[col] = merged[col].interpolate(limit_direction='both').fillna(0)

    # anchor / eval_zone を特定
    anchor    = hw_test[hw_test['TVT_input'].notna()]
    eval_zone = hw_test[hw_test['TVT_input'].isna()]

    if len(anchor) == 0 or len(eval_zone) == 0:
        print(f'{well_id}: anchor={len(anchor)}, eval={len(eval_zone)} → skip')
        continue

    last    = anchor.iloc[-1]
    la_tvt  = float(last['TVT_input'])
    la_z    = float(last['Z'])
    la_ancc = float(last['ANCC']) if 'ANCC' in last.index else 0.0

    print(f'{well_id}: anchor_tvt={la_tvt:.2f}, anchor_Z={la_z:.2f}, anchor_ANCC={la_ancc:.2f}')

    if not np.isfinite(la_ancc) or la_ancc == 0.0:
        print(f'  WARNING: ANCC invalid → falling back to anchor TVT')
        f_tvt = np.full(len(eval_zone), la_tvt)
    else:
        b = la_tvt + la_z - la_ancc
        f_tvt = b - eval_zone['Z'].values + eval_zone['ANCC'].values
        f_tvt = np.where(np.isfinite(f_tvt), f_tvt, la_tvt)

    print(f'  form_tvt: [{f_tvt.min():.2f}, {f_tvt.max():.2f}], mean={f_tvt.mean():.2f}')

    for idx_row, tvt in zip(eval_zone.index, f_tvt):
        all_preds[f'{well_id}_{idx_row}'] = tvt

sub['tvt'] = sub['id'].map(all_preds)
print(f'\nNull count: {sub["tvt"].isnull().sum()}')
print(sub.describe())
sub.to_csv('submission.csv', index=False)
print('\nSaved: submission.csv')